# Oslofjorden Bathymetry — GeoTIFF Exploration

Parsing and visualizing depth data from `Dybde_Oslo_1m.tif` (1m resolution bathymetry).

In [ ]:
import sys
sys.path.insert(0, '../..')
from project_paths import REPO_ROOT

import rasterio
import numpy as np
import matplotlib.pyplot as plt
from rasterio.enums import Resampling
from rasterio.warp import transform_bounds

TIFF_PATH = str(REPO_ROOT / 'Datasets/Dybde_Oslofjorden/Dybde_Oslo_1m.tif')

## 1. Inspect metadata

In [ ]:
with rasterio.open(TIFF_PATH) as src:
    print(f"CRS:        {src.crs}")
    print(f"Bounds:     {src.bounds}")
    print(f"Shape:      {src.shape}  (rows x cols)")
    print(f"Resolution: {src.res}  m/pixel")
    print(f"NoData:     {src.nodata}")
    print(f"Dtype:      {src.dtypes}")
    print(f"Overviews:  {src.overviews(1)}")

## 2. Read downsampled (10x decimation)

In [ ]:
DOWNSAMPLE = 10

with rasterio.open(TIFF_PATH) as src:
    # Read at reduced resolution
    out_shape = (src.height // DOWNSAMPLE, src.width // DOWNSAMPLE)
    depth = src.read(
        1,
        out_shape=out_shape,
        resampling=Resampling.average,
    )
    # Build coordinate arrays in UTM
    transform = src.transform * src.transform.scale(
        src.width / out_shape[1],
        src.height / out_shape[0],
    )
    nodata = src.nodata
    crs = src.crs
    bounds = src.bounds

# Mask nodata
depth = depth.astype(np.float32)
depth[depth <= -1e30] = np.nan

print(f"Downsampled shape: {depth.shape}")
print(f"Depth range: {np.nanmin(depth):.1f} m  to  {np.nanmax(depth):.1f} m")
print(f"Valid pixels: {np.count_nonzero(~np.isnan(depth)):,} / {depth.size:,}  ({100*np.count_nonzero(~np.isnan(depth))/depth.size:.1f}%)")

## 3. Bathymetry heatmap

In [ ]:
# UTM coordinate arrays for axes
rows, cols = depth.shape
x = np.linspace(bounds.left, bounds.right, cols)
y = np.linspace(bounds.top, bounds.bottom, rows)  # top-to-bottom

fig, ax = plt.subplots(figsize=(12, 16))

# Only show underwater (negative = below sea level)
depth_display = depth.copy()
depth_display[depth_display > 0] = np.nan

im = ax.pcolormesh(
    x, y, depth_display,
    cmap="Blues_r",
    vmin=-180, vmax=0,
    shading="auto",
    rasterized=True,
)
cbar = fig.colorbar(im, ax=ax, label="Depth (m)", shrink=0.6)
ax.set_xlabel("Easting (m, UTM 32N)")
ax.set_ylabel("Northing (m, UTM 32N)")
ax.set_title("Oslofjorden Bathymetry")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## 4. Depth histogram

In [ ]:
valid_depths = depth[~np.isnan(depth)]
# Only underwater values
underwater = valid_depths[valid_depths < 0]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(underwater, bins=200, color="steelblue", edgecolor="none")
ax.set_xlabel("Depth (m)")
ax.set_ylabel("Pixel count")
ax.set_title(f"Depth distribution (n={len(underwater):,} pixels)")
ax.axvline(-10, color="red", ls="--", alpha=0.5, label="-10m")
ax.axvline(-50, color="orange", ls="--", alpha=0.5, label="-50m")
ax.axvline(-100, color="purple", ls="--", alpha=0.5, label="-100m")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Depth contour map in lat/lon

In [ ]:
# Transform bounds to WGS84 lat/lon
lon_min, lat_min, lon_max, lat_max = transform_bounds(crs, "EPSG:4326", *bounds)
print(f"Lat: {lat_min:.4f} – {lat_max:.4f}")
print(f"Lon: {lon_min:.4f} – {lon_max:.4f}")

lons = np.linspace(lon_min, lon_max, cols)
lats = np.linspace(lat_max, lat_min, rows)  # top-to-bottom

fig, ax = plt.subplots(figsize=(12, 16))

# Background heatmap
im = ax.pcolormesh(
    lons, lats, depth_display,
    cmap="Blues_r",
    vmin=-180, vmax=0,
    shading="auto",
    rasterized=True,
)

# Contour lines at key depths
contour_levels = [-150, -100, -50, -10]
cs = ax.contour(
    lons, lats, depth_display,
    levels=contour_levels,
    colors="white",
    linewidths=0.8,
    linestyles="solid",
)
ax.clabel(cs, fmt="%d m", fontsize=8, colors="white")

fig.colorbar(im, ax=ax, label="Depth (m)", shrink=0.6)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Oslofjorden Bathymetry (WGS84)")
ax.set_aspect(1 / np.cos(np.radians((lat_min + lat_max) / 2)))  # Mercator-like correction
plt.tight_layout()
plt.show()

## 6. Zoomed view — deepest area

In [ ]:
# Find deepest point and zoom in around it
min_idx = np.unravel_index(np.nanargmin(depth_display), depth_display.shape)
deep_lat = lats[min_idx[0]]
deep_lon = lons[min_idx[1]]
print(f"Deepest point: {np.nanmin(depth_display):.1f} m at ({deep_lat:.4f}°N, {deep_lon:.4f}°E)")

# Zoom window ~2km around deepest point
dlat = 0.015
dlon = 0.03

fig, ax = plt.subplots(figsize=(10, 10))
im = ax.pcolormesh(
    lons, lats, depth_display,
    cmap="Blues_r",
    vmin=-180, vmax=0,
    shading="auto",
    rasterized=True,
)
ax.plot(deep_lon, deep_lat, "r*", markersize=15, label=f"Deepest: {np.nanmin(depth_display):.0f}m")
ax.set_xlim(deep_lon - dlon, deep_lon + dlon)
ax.set_ylim(deep_lat - dlat, deep_lat + dlat)
ax.set_aspect(1 / np.cos(np.radians(deep_lat)))
fig.colorbar(im, ax=ax, label="Depth (m)", shrink=0.6)
ax.legend(fontsize=12)
ax.set_title("Zoomed: Deepest area in Oslofjorden")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()